In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord
import astropy.units as u
import logging


In [2]:

def setup_logging(verbose=False):
    """
    Call this once at the top of the notebook.
    If verbose=True, DEBUG messages will show; otherwise only INFO+.
    """
    name = "Coma_GCs_debug"
    
    # Remove any root handlers (if I had root-level logging before)
    for h in logging.root.handlers[:]:
        logging.root.removeHandler(h)

    # Get named logger and clear *its* handlers + disable propagation
    logger = logging.getLogger(name)
    for h in logger.handlers[:]:
        logger.removeHandler(h)
    logger.propagate = False

    # Set logger’s level
    logger.setLevel(logging.DEBUG)   # we capture everything here

    # INFO-only handler (no timestamp)
    info_handler = logging.StreamHandler()
    info_handler.setLevel(logging.INFO)
    info_handler.addFilter(lambda rec: rec.levelno == logging.INFO)
    info_handler.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(info_handler)

    # non-INFO handler (timestamped, DEBUG or WARNING+)
    other = logging.StreamHandler()
    other.setLevel(logging.DEBUG if verbose else logging.WARNING)
    other.addFilter(lambda rec: rec.levelno != logging.INFO)
    # allow only your logger’s DEBUG (if you still want a name-filter):
    other.addFilter(lambda rec: rec.levelno != logging.DEBUG or rec.name == name)
    fmt = "[%(levelname)s] %(asctime)s.%(msecs)03d %(message)s"
    other.setFormatter(logging.Formatter(fmt, datefmt="%H:%M:%S"))
    logger.addHandler(other)


In [3]:

# Set verbosity here:
setup_logging(verbose=True)   # or False

# testing logging
logger = logging.getLogger("Coma_GCs_debug")
logger.debug("Debug messages are ON")
logger.info("Info messages are always shown")
logger.warning("Warnings also show up")


[DEBUG] 14:36:42.919 Debug messages are ON
Info messages are always shown
[WARNING] 14:36:42.920 Warnings also show up


## Load in galaxies

Load in a `csv` file, which has been produced from manual and online archives, and saved in `Coma_Gal_List_Create.ipynb`, so this can be used in the calculation and plotting routines.

A second file of more data from the SIMBAD and NED archives has bee extracted, but this has indeterminate or somewhat inconsistent effective radii.

Select using `archive = True` or `False` if manual data is preferred.


In [4]:
archive = True
if archive:
    logger.warning('NOTE: Loading SIMBAD / NED extracted data for Coma cluster galaxies')
    # Load in the file of galaxy data extracted from archives OR 
    gals_df = pd.read_csv('../Coma_LSD_CSS/data/gals_data_from_archives.csv')
else:
    logger.warning('NOTE: Loading manual data for Coma cluster galaxies')
    # Load in the file of sanitised and augmented galaxy data  
    gals_df = pd.read_csv('../Coma_LSD_CSS/data/gals_data_cleaned.csv')
gals_df.tail(5)

[WARNING] 14:36:46.365 NOTE: Loading SIMBAD / NED extracted data for Coma cluster galaxies


,name,ra,dec,otype,z,mv,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV,show_label,Re_arcsec
169,LEDA 126761,195.077274,28.097154,G,0.02608,17.040001,15.108181,0.185831,172.0,SIMBAD,B,2023ApJS..269....3M,NaN,-17.959999,False,1.888523
170,2MASX J12591389+2804349,194.808025,28.076248,GiC,0.02605,14.890000,42.662220,0.687217,72.0,SIMBAD,B,2023ApJS..269....3M,NaN,-20.110000,False,5.332778
171,IC 3973,194.878430,27.884206,AG?,0.01571,14.030000,49.606979,0.516492,137.0,SIMBAD,B,2023ApJS..269....3M,NaN,-20.970000,True,6.200872
172,Mrk 60,195.038088,27.866484,AGN,0.01773,15.510000,22.353781,0.261167,169.0,SIMBAD,B,2023ApJS..269....3M,NaN,-19.490000,False,2.794223
173,Coma cluster,195.017071,27.977025,CLUSTER,0.02150,NaN,NaN,NaN,NaN,MANUAL,NaN,NaN,NaN,NaN,False,NaN


## Galaxy type analysis

Count each SIMBAD otype in the loaded DataFrame and attach a short human-readable meaning for the common galaxy codes.


In [6]:

# per-type counts
otype_counts = (
    gals_df["otype"].astype("string").str.strip()
      .value_counts(dropna=False)
      .rename_axis("otype")
      .reset_index(name="count")
)
otype_counts.head(10)


,otype,count
0,GiC,62
1,G,52
2,AG?,25
3,EmG,16
4,LIN,5
5,AGN,5
6,rG,3
7,GiG,3
8,LSB,2
9,CLUSTER,1


In [7]:
# minimal, practical cheat-sheet (not exhaustive)
otype_map = {
    # single galaxies
    "G": "Galaxy",
    "G?": "Galaxy candidate",
    "LSB": "Low-surface-brightness galaxy",
    "BCD": "Blue compact dwarf galaxy",
    "dG": "Dwarf galaxy",
    "IG": "Interacting galaxy",
    "EmG": "Emission-line galaxy",
    # activity/AGN subclasses
    "SyG": "Seyfert galaxy",
    "Sy1": "Seyfert 1",
    "Sy2": "Seyfert 2",
    "LIN": "LINER galaxy",
    "AGN": "Active galactic nucleus (galaxy)",
    "BLLac": "BL Lac object",
    "QSO": "Quasar",
    # membership in larger structures
    "GiP": "Galaxy in a pair",
    "GiG": "Galaxy in a group",
    "GiC": "Galaxy in a cluster",
    "BiC": "Brightest cluster galaxy (BCG)",
    # systems of galaxies
    "PaG": "Pair of galaxies",
    "GrG": "Group of galaxies",
    "CGG": "Compact group of galaxies",
    "ClG": "Cluster of galaxies",
    "SCG": "Supercluster of galaxies",
    # broad morphology (when present in otype)
    "E": "Elliptical galaxy",
    "S0": "Lenticular galaxy",
    "S": "Spiral galaxy",
    "SA": "Unbarred spiral",
    "SB": "Barred spiral",
    "SAB": "Weakly barred spiral",
    "Im": "Irregular galaxy",
    # alternates you might also see
    "GPair": "Galaxy pair",
    "GTrpl": "Galaxy triple",
    "GGroup": "Galaxy group",
}

lookup = (pd.DataFrame.from_dict(otype_map, orient="index", columns=["meaning"])
          .reset_index().rename(columns={"index": "otype"}))

otype_counts = otype_counts.merge(lookup, on="otype", how="left")
otype_counts.head(20)


,otype,count,meaning
0,GiC,62,Galaxy in a cluster
1,G,52,Galaxy
2,AG?,25,NaN
3,EmG,16,Emission-line galaxy
4,LIN,5,LINER galaxy
5,AGN,5,Active galactic nucleus (galaxy)
6,rG,3,NaN
7,GiG,3,Galaxy in a group
8,LSB,2,Low-surface-brightness galaxy
9,CLUSTER,1,NaN


In [10]:
family_bins = {
    "Active (AGN/Seyfert/QSO)": {"AGN","SyG","Sy1","Sy2","LIN","BLLac","QSO"},
    "Systems (pairs/groups/clusters)": {"PaG","GrG","CGG","ClG","SCG","GPair","GTrpl","GGroup"},
    "Members of systems": {"GiP","GiG","GiC","BiC"},
    "LSB/BCD/Dwarf": {"LSB","BCD","dG"},
    "Regular galaxies": {"G","E","S0","S","SA","SB","SAB","Im","IG","EmG"},
}
rev = {code: fam for fam, codes in family_bins.items() for code in codes}
gals_df["otype_family"] = gals_df["otype"].map(rev).fillna("Other")

family_counts = (gals_df["otype_family"]
                 .value_counts()
                 .rename_axis("otype_family")
                 .reset_index(name="count"))
family_counts


,otype_family,count
0,Regular galaxies,68
1,Members of systems,65
2,Other,29
3,Active (AGN/Seyfert/QSO),10
4,LSB/BCD/Dwarf,2


In [25]:
from astroquery.simbad import Simbad
s = Simbad(); 
s.add_votable_fields("morphtype")

morph = s.query_objects(gals_df["name"].tolist()).to_pandas()[["main_id","morph_type"]] 
morph = morph.rename(columns={"main_id": "name"})
morph
gals_df = gals_df.merge(morph, on="name", how="left")


In [26]:
is_E   = (gals_df["otype"] == "E")
is_BCG = (gals_df["otype"] == "BiC")  # SIMBAD code for brightest cluster galaxy
has_cD = gals_df["morph_type"].astype("string").str.contains(r"\bcD\b", na=False)


In [28]:
from astropy.cosmology import Planck15 as cosmo
import numpy as np

# distance modulus & angular diameter distance
z = gals_df["z"].astype(float)
DM = cosmo.distmod(z).value                 # mag
DM = 35.0  # assumed value 
DA = cosmo.angular_diameter_distance(z).to("kpc").value  # kpc

# absolute M_V (ignore extinction unless you have it)
# gals_df["M_V"] = gals_df["Vmag"].astype(float) - DM

# crude physical size from your major-axis measurement if present (arcsec → kpc)
maj_arcsec = pd.to_numeric(gals_df.get("major_axis_arcsec"), errors="coerce")
gals_df["major_kpc"] = (maj_arcsec/206265.0) * (DA*1e3)  # 1 rad = 206265"

is_luminous = gals_df["MV"] <= -21.5
is_huge     = gals_df["major_kpc"] >= 30                  # very extended on the sky


In [31]:
gals_df[is_luminous]


,name,ra,dec,otype,z,mv,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV,show_label,Re_arcsec,otype_family,morph_type,major_kpc
23,NGC 4889,195.033738,27.977025,EmG,0.02150,11.30,175.404602,1.816900,75.0,SIMBAD,B,2023ApJS..269....3M,NaN,-23.70,True,21.925575,Regular galaxies,NaN,78815.629278
33,NGC 4874,194.898789,27.959248,LIN,0.02391,12.71,140.404205,2.260740,63.0,SIMBAD,B,2023ApJS..269....3M,NaN,-22.29,True,17.550526,Active (AGN/Seyfert/QSO),NaN,69955.709614
100,IC 4051,195.226929,28.007639,EmG,0.01662,13.50,79.127998,0.915775,103.0,SIMBAD,B,2023ApJS..269....3M,NaN,-21.50,True,9.891000,Regular galaxies,E3,27648.455901


In [32]:
gals_df[is_huge]


,name,ra,dec,otype,z,mv,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV,show_label,Re_arcsec,otype_family,morph_type,major_kpc
0,LEDA 3098454,195.075410,27.956592,GiC,0.021240,14.333000,23.714220,0.229277,143.0,SIMBAD,B,2023ApJS..269....3M,NaN,-20.667000,False,2.964278,Members of systems,SB0:,10530.120364
1,SDSS J125939.47+275116.5,194.914488,27.854598,G,0.031340,16.944000,1.300000,NaN,NaN,NED,NaN,NaN,2007SDSS6.C...0000:,-18.056000,False,0.162500,Regular galaxies,,841.404907
2,LEDA 126763,195.055833,28.053417,GiC,0.027160,17.309999,23.923620,0.110128,81.0,SIMBAD,B,2023ApJS..269....3M,NaN,-17.690001,False,2.990453,Members of systems,NaN,13486.880114
3,GMP 3424,194.872189,27.942225,GiC,0.018723,18.773001,4.070000,NaN,NaN,NED,NaN,NaN,2007SDSS6.C...0000:,-16.226999,False,0.508750,Members of systems,dE,1597.967878
4,LEDA 126771,195.015438,27.964525,GiC,0.017780,17.260000,7.150000,NaN,NaN,NED,NaN,NaN,2007SDSS6.C...0000:,-17.740000,False,0.893750,Members of systems,NaN,2668.911659
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,IC 4021,195.061455,28.041300,LIN,0.019120,14.750000,25.321800,0.362693,49.0,SIMBAD,B,2023ApJS..269....3M,NaN,-20.250000,True,3.165225,Active (AGN/Seyfert/QSO),E3,10147.779410
169,LEDA 126761,195.077274,28.097154,G,0.026080,17.040001,15.108181,0.185831,172.0,SIMBAD,B,2023ApJS..269....3M,NaN,-17.959999,False,1.888523,Regular galaxies,NaN,8189.209492
170,2MASX J12591389+2804349,194.808025,28.076248,GiC,0.026050,14.890000,42.662220,0.687217,72.0,SIMBAD,B,2023ApJS..269....3M,NaN,-20.110000,False,5.332778,Members of systems,E1,23098.786248
171,IC 3973,194.878430,27.884206,AG?,0.015710,14.030000,49.606979,0.516492,137.0,SIMBAD,B,2023ApJS..269....3M,NaN,-20.970000,True,6.200872,Other,SBa,16402.501923


In [33]:
gals_df["is_large_E"] = (is_E & (is_luminous | is_huge)) | is_BCG | has_cD


In [34]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord, match_coordinates_sky
import astropy.units as u
from astroquery.sdss import SDSS

def attach_sdss_re(gals_df, ra_col="ra", dec_col="dec", band="r", radius="3 arcsec", dr=17):
    df = gals_df.copy()
    coords = SkyCoord(df[ra_col].values, df[dec_col].values, unit='deg', frame='icrs')

    # Ask SDSS CrossID for the size fields we need (one nearest primary per input by default)
    fields = [
        f"deVRad_{band}", f"deVAB_{band}",  # de Vauc effective radius & b/a
        f"expRad_{band}", f"expAB_{band}",  # exponential effective radius & b/a (SDSS defines expRad as Re)
        f"fracDeV_{band}",                  # fraction of deV in the cModel (decision helper)
        "type", "mode", "objid", "ra", "dec"
    ]
    xid = SDSS.query_crossid(coords, radius=radius, photoobj_fields=fields, data_release=dr)
    if xid is None or len(xid) == 0:
        # nothing matched; just return original df with empty columns
        for c in ["Re_r_arcsec", "Re_r_arcsec_circ", "sdss_objid"]:
            df[c] = pd.NA
        return df

    t = xid.to_pandas()

    # Prefer primary galaxy detections if columns exist
    if "mode" in t.columns:
        t = t[t["mode"] == 1]
    if "type" in t.columns:
        t = t[t["type"] == 3]  # 3=GALAXY

    # Match returned SDSS rows back to input rows by nearest on-sky
    sdss_sc = SkyCoord(t["ra"].values, t["dec"].values, unit='deg')
    idx, sep, _ = match_coordinates_sky(coords, sdss_sc)
    # Trust matches within the requested radius
    max_sep = u.Quantity(radius)
    good = sep <= max_sep

    # Prepare result series aligned to input
    sel = t.iloc[idx].reset_index(drop=True)
    sel.loc[~good, :] = np.nan  # blank out non-matches

    # Choose an Re per object:
    # - SDSS defines deVRad_* as half-light (effective) radius for de Vaucouleurs fits
    # - SDSS defines expRad_* as half-light (effective) radius for exponential fits
    frac = sel.get(f"fracDeV_{band}")
    Re_deV = sel.get(f"deVRad_{band}")
    Re_exp = sel.get(f"expRad_{band}")

    use_deV = frac.notna() & (frac >= 0.5)
    Re = np.where(use_deV, Re_deV, Re_exp)

    # Circularize using the matching b/a
    ba = np.where(use_deV, sel.get(f"deVAB_{band}"), sel.get(f"expAB_{band}"))
    Re_circ = Re * np.sqrt(ba)

    # Attach to your DataFrame
    df["sdss_objid"]       = sel.get("objid").values
    df["Re_r_arcsec"]      = pd.to_numeric(Re, errors="coerce")
    df["Re_r_arcsec_circ"] = pd.to_numeric(Re_circ, errors="coerce")

    return df



In [35]:
gals_df = attach_sdss_re(gals_df, ra_col="ra", dec_col="dec", band="r", radius="3 arcsec", dr=17)


In [36]:
gals_df


,name,ra,dec,otype,z,mv,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,...,MV,show_label,Re_arcsec,otype_family,morph_type,major_kpc,is_large_E,sdss_objid,Re_r_arcsec,Re_r_arcsec_circ
0,LEDA 3098454,195.075410,27.956592,GiC,0.021240,14.333000,23.714220,0.229277,143.0,SIMBAD,...,-20.667000,False,2.964278,Members of systems,SB0:,10530.120364,False,1.237667e+18,4.745432,4.300019
1,SDSS J125939.47+275116.5,194.914488,27.854598,G,0.031340,16.944000,1.300000,NaN,NaN,NED,...,-18.056000,False,0.162500,Regular galaxies,,841.404907,False,1.237667e+18,7.100106,4.281512
2,LEDA 126763,195.055833,28.053417,GiC,0.027160,17.309999,23.923620,0.110128,81.0,SIMBAD,...,-17.690001,False,2.990453,Members of systems,NaN,13486.880114,False,1.237667e+18,5.577060,2.565344
3,GMP 3424,194.872189,27.942225,GiC,0.018723,18.773001,4.070000,NaN,NaN,NED,...,-16.226999,False,0.508750,Members of systems,dE,1597.967878,False,1.237667e+18,0.765117,0.692910
4,LEDA 126771,195.015438,27.964525,GiC,0.017780,17.260000,7.150000,NaN,NaN,NED,...,-17.740000,False,0.893750,Members of systems,NaN,2668.911659,False,1.237667e+18,9.422453,7.926311
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,LEDA 126761,195.077274,28.097154,G,0.026080,17.040001,15.108181,0.185831,172.0,SIMBAD,...,-17.959999,False,1.888523,Regular galaxies,NaN,8189.209492,False,1.237667e+18,2.984147,2.739612
170,2MASX J12591389+2804349,194.808025,28.076248,GiC,0.026050,14.890000,42.662220,0.687217,72.0,SIMBAD,...,-20.110000,False,5.332778,Members of systems,E1,23098.786248,False,1.237667e+18,14.073110,13.508782
171,IC 3973,194.878430,27.884206,AG?,0.015710,14.030000,49.606979,0.516492,137.0,SIMBAD,...,-20.970000,True,6.200872,Other,SBa,16402.501923,False,1.237667e+18,3.882846,3.346397
172,Mrk 60,195.038088,27.866484,AGN,0.017730,15.510000,22.353781,0.261167,169.0,SIMBAD,...,-19.490000,False,2.794223,Active (AGN/Seyfert/QSO),NaN,8321.134612,False,1.237667e+18,3.132543,2.653063


In [40]:
gals_df.sort_values(by='Re_r_arcsec_circ', ascending=False).head(30)


,name,ra,dec,otype,z,mv,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,...,MV,show_label,Re_arcsec,otype_family,morph_type,major_kpc,is_large_E,sdss_objid,Re_r_arcsec,Re_r_arcsec_circ
33,NGC 4874,194.898789,27.959248,LIN,0.023910,12.710000,140.404205,2.260740,63.0,SIMBAD,...,-22.290000,True,17.550526,Active (AGN/Seyfert/QSO),NaN,69955.709614,False,1.237667e+18,29.679890,29.586330
23,NGC 4889,195.033738,27.977025,EmG,0.021500,11.300000,175.404602,1.816900,75.0,SIMBAD,...,-23.700000,True,21.925575,Regular galaxies,NaN,78815.629278,False,1.237667e+18,29.678560,23.026233
60,LEDA 44708,195.025448,27.978315,GiC,0.025460,15.994000,33.000000,0.190000,141.0,SIMBAD,...,-19.006000,False,4.125000,Members of systems,NaN,17475.131768,False,1.237667e+18,29.675960,21.289626
16,SDSS J130051.15+280249.7,195.213151,28.047139,GiC,0.021160,17.077000,15.450000,NaN,NaN,NED,...,-17.923000,False,1.931250,Members of systems,,6835.279943,False,1.237667e+18,16.102630,14.125548
100,IC 4051,195.226929,28.007639,EmG,0.016620,13.500000,79.127998,0.915775,103.0,SIMBAD,...,-21.500000,True,9.891000,Regular galaxies,E3,27648.455901,False,1.237667e+18,16.322610,14.110691
170,2MASX J12591389+2804349,194.808025,28.076248,GiC,0.026050,14.890000,42.662220,0.687217,72.0,SIMBAD,...,-20.110000,False,5.332778,Members of systems,E1,23098.786248,False,1.237667e+18,14.073110,13.508782
142,SDSS J130042.56+280658.6,195.177360,28.116320,LSB,0.020701,18.240000,19.498001,0.324967,0.0,SIMBAD,...,-16.760000,False,2.437250,LSB/BCD/Dwarf,,8443.755209,False,1.237667e+18,11.911510,10.328826
50,LEDA 44636,194.909672,27.987209,G,0.022740,15.890000,16.380001,0.191000,40.0,SIMBAD,...,-19.110000,False,2.047500,Regular galaxies,NaN,7772.914363,False,1.237667e+18,11.870460,9.669536
136,LEDA 126768,195.031143,27.958069,GiC,0.020720,17.580000,13.800000,0.210000,NaN,SIMBAD,...,-17.420000,False,1.725000,Members of systems,NaN,5981.540402,False,1.237667e+18,9.742112,8.978572
122,LEDA 126762,195.056811,27.867177,GiC,0.024680,16.350000,18.000000,0.210000,109.0,SIMBAD,...,-18.650000,False,2.250000,Members of systems,NaN,9248.599687,False,1.237667e+18,9.405168,8.298744


In [41]:
def attach_gc_Re_defaults(gals_df):
    df = gals_df.copy()
    # pick your galaxy Re (prefer circularized)
    Re_gal_arcsec = df.get("Re_r_arcsec_circ").fillna(df.get("Re_r_arcsec"))
    # scaling factor by class
    scale = np.where(df.get("is_large_E", False), 4.8, np.where(df["otype"].eq("E"), 3.2, 3.0))
    df["Re_GCS_arcsec"] = Re_gal_arcsec.astype(float) * scale

    return df

gals_df = attach_gc_Re_defaults(gals_df)


In [42]:
gals_df.sort_values(by='Re_r_arcsec_circ', ascending=False).head(30)


,name,ra,dec,otype,z,mv,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,...,show_label,Re_arcsec,otype_family,morph_type,major_kpc,is_large_E,sdss_objid,Re_r_arcsec,Re_r_arcsec_circ,Re_GCS_arcsec
33,NGC 4874,194.898789,27.959248,LIN,0.023910,12.710000,140.404205,2.260740,63.0,SIMBAD,...,True,17.550526,Active (AGN/Seyfert/QSO),NaN,69955.709614,False,1.237667e+18,29.679890,29.586330,88.758989
23,NGC 4889,195.033738,27.977025,EmG,0.021500,11.300000,175.404602,1.816900,75.0,SIMBAD,...,True,21.925575,Regular galaxies,NaN,78815.629278,False,1.237667e+18,29.678560,23.026233,69.078698
60,LEDA 44708,195.025448,27.978315,GiC,0.025460,15.994000,33.000000,0.190000,141.0,SIMBAD,...,False,4.125000,Members of systems,NaN,17475.131768,False,1.237667e+18,29.675960,21.289626,63.868877
16,SDSS J130051.15+280249.7,195.213151,28.047139,GiC,0.021160,17.077000,15.450000,NaN,NaN,NED,...,False,1.931250,Members of systems,,6835.279943,False,1.237667e+18,16.102630,14.125548,42.376644
100,IC 4051,195.226929,28.007639,EmG,0.016620,13.500000,79.127998,0.915775,103.0,SIMBAD,...,True,9.891000,Regular galaxies,E3,27648.455901,False,1.237667e+18,16.322610,14.110691,42.332073
170,2MASX J12591389+2804349,194.808025,28.076248,GiC,0.026050,14.890000,42.662220,0.687217,72.0,SIMBAD,...,False,5.332778,Members of systems,E1,23098.786248,False,1.237667e+18,14.073110,13.508782,40.526346
142,SDSS J130042.56+280658.6,195.177360,28.116320,LSB,0.020701,18.240000,19.498001,0.324967,0.0,SIMBAD,...,False,2.437250,LSB/BCD/Dwarf,,8443.755209,False,1.237667e+18,11.911510,10.328826,30.986478
50,LEDA 44636,194.909672,27.987209,G,0.022740,15.890000,16.380001,0.191000,40.0,SIMBAD,...,False,2.047500,Regular galaxies,NaN,7772.914363,False,1.237667e+18,11.870460,9.669536,29.008607
136,LEDA 126768,195.031143,27.958069,GiC,0.020720,17.580000,13.800000,0.210000,NaN,SIMBAD,...,False,1.725000,Members of systems,NaN,5981.540402,False,1.237667e+18,9.742112,8.978572,26.935717
122,LEDA 126762,195.056811,27.867177,GiC,0.024680,16.350000,18.000000,0.210000,109.0,SIMBAD,...,False,2.250000,Members of systems,NaN,9248.599687,False,1.237667e+18,9.405168,8.298744,24.896231
